<a href="https://colab.research.google.com/github/AmirGeo/otbasybank_internship/blob/main/ServiceDesk_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xlrd

import pandas as pd

q1 = pd.read_excel('/content/1 кв.xls',header=6)
q2 = pd.read_excel('/content/2кв.xls',header=6)
q3 = pd.read_excel('/content/3кв.xls',header=6)
q4 = pd.read_excel('/content/4 кв декабрь.xls',header=6)
q5 = pd.read_excel('/content/4 кв октябрь-ноябрь.xls',header=6)

df = pd.concat([q1,q2,q3,q4,q5].ignore_index=True)

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 2','Unnamed: 5'])

df.columns

In [ ]:
print(df['Статус'].value_counts())

In [ ]:
import re

iin_pattern = re.compile(
    r'(?ИИН|ЖСН)\s*[:\-]?\s*\d{12}',
    re.IGNORECASE
)

raw_12_digit_pattern = re.compile(r'\b\d{12}\b')

iban_pattern = re.compile(
    r'\bKZ\d{17,21}\b',
    re.IGNORECASE
)

account_pattern =  re.compile(
    r'сч[её]т\s*:?\s*KZ\d{17,21}',
    re.IGNORECASE
)

kz_account_pattern = re.compile(
    r'bKZ[A-Z0-9]{10,}\b',
    re.IGNORECASE
)

def normalize_text(value):
  if not isinstance(value,str):
    return value

  value = iin_pattern.sub('', value)
  value = raw_12_digit_pattern.sub('', value)
  value = account_pattern.sub('', value)
  value = iban_pattern.sub('', value)
  value = kz_account_pattern.sub('', value)

  value = re.sub(r'\s+',' ', value)

  return value.strip()

df['Описание'] = df['Описание'].apply(normalize_text)

In [ ]:
df.dropna(subset=['Описание'],how='all',inplace=True)

df['full_text'] = (
    'Тип заявки: ' + df['Тип заявки'] + '. '+
    'Программное обеспечение: ' + df['Программное обеспечение'] + '. '+
    'Наименование: ' + df['Наименование']+ '. ' +
    'Описание: ' + df['Описание']
)

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence-transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

texts = df['full_text'].tolist()
embedding_texts = [
    f"query: {text}" for text in texts
]

embedding_model = SentenceTransformer('intfloat/multilingual-e5-base',device=device)
vectors = embedding_model.encode(embedding_texts,batch_size=128, show_progress_bar=True)
vectors = normalize(vectors)
np.save('/content/embeddings.npy',vectors)

In [ ]:
import numpy as np

vectors = np.load('/content/embeddings.npy')

!pip install bertopic

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import torch

texts = df['full_text'].tolist()

umap_model = UMAP(n_components=10,min_dist=0.0,metric='cosine',random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=150,min_samples=50,metric='euclidean',cluster_selection_method='eom',prediction_data=True)
embedding_model = SentenceTransformer('intfloat/multilingual-e5-base',device=device)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

topics,probs = topic_model.fit_transform(
    texts,
    embeddings=vectors,
)

In [ ]:
topic_info = topic_model.get_topic_info()

topic_keywords = {}
for topic_id in topic_info['topic']:
  if topic_id == -1:
    topic_keywords[-1] = ""
  else:
    words = topic_model.get_topic(topic_id)
    topic_keywords[topic_id] = "|".join([w for w, _ in words[:3]])

df['Keywords'] = [topic_keywords[t] for t in topics]

In [ ]:
df['label'] = [0 if t==-1 else 1 for t in topics]

df['label'] = 1

unprocessable_comments = [
    'внятн','не смог разобрать','непонят',
    'уточнит','не понял','некоррект','не указан',
    'вложите','что за','понятнее','не понятн',
    'не указан','подробнее'
]

comment_pattern = '|'.join(unprocessable_comments)

unprocessable_mask = df['Комментарий исполнителя'].str.contains(comment_pattern,case=False,regex=True,na=False)

df.loc[unprocessable_mask,'label'] = 0

print(f"Unprocessable via comment: {unprocessble_mask.sum()}")
print(df['label'].value_counts())

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

svm = LinearSVC(class_weight='balanced',random_state=42,loss="hinge",fit_intercept=False)
svm.fit(X_train,y_train)
y_pred = svm.predict(X_test)

print(classification_report(y_test,y_pred, target_names=['Gibberish(0), Processable(1)']))

cm = confusion_matrix(y_test,y_pred)
ConfusionMatrixDisplay(cm,display_labels=['Gibberish(0), Processable(1)']).plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42, stratify=y)

pipeline = Pipeline([
    ('smote',SMOTE(random_state=42)),
    ('svm',LinearSVC(class_weight='balanced',random_state=42,loss="hinge",fit_intercept=False))
])

pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test,y_pred,target_names=['Gibberish(0)','Processable(1)']))

cm  = confusion_matrix(y_test,y_pred)
ConfusionMatrixDisplay(cm,display_labels=['Gibberish(0)','Processable(1)']).plot(cmap='Blues')
plt.show()

In [ ]:
df['Группа поддержки'].value_counts()

df['topics'] = topics
df_routing = df[df['Группа поддержки']!='Не выбран'].reset_index(drop=True)

cross_tab = pd.crosstab(
    df_routing['topics'],
    df_routing['Группа поддержки'],
    normalize='index'
)
print(cross_tab.round(2))

dominant_group = cross_tab.idxmax(axis=1)
dominant_share = cross_tab.max(axis=1)

topic_routing = pd.DataFrame({
    'dominant_group':dominant_group,
    'share': dominant_share
})
print(topic_routing.sort_values('share',ascending=False).to_string())

print(f"Min share: {dominant_share.min():.2f}")
print(f"Max share: {dominant_share.max():.2f}")
print(f"Mean share: {dominant_share.mean():.2f}")
print(f"50%: {(dominant_share>0.5).sum()}")
print(f"70%: {(dominant_share>0.70).sum()}")
print(f"Dom share: {len(dominant_share)}")

In [ ]:
print(df_routing[df_routing['topics']==0]['Группа поддержки'].value_counts())

topic_to_group = dominant_group.to_dict()

topic_to_group[0] = 'Первая линия'
df_routing['predicted_group'] = df_routing['topics'].map(topic_to_group)

print(df_routing['predicted_group'].value_counts())

print(classification_report(
    df_routing['Группа поддержки'],
    df_routing['predicted_group'],
    target_names=dominant_group['Группа поддержки'].unique()
))

comparison = pd.DataFrame({
    'actual':df_routing['Группа поддержки'].value_counts(),
    'predicted':df_routing['predicted_group'].value_counts()
})

print(comparison)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

topic_to_group = dominant_group.to_dict()
topic_to_group[-1] = 'ДБПиТ'
df_routing['predicted_group'] = df_routing['topics'].map(topic_to_group)
df_routing['predicted_group'] = df_routing['predicted_group'].fillna(df_routing['Группа поддержки'])

df_routing['predicted_before_knn'] = df_routing['predicted_group'].copy()

mask_not_noise = df_routing['topics'] != -1
mask_noise = df_routing['topics'] == -1

X_known = vectors[df_routing.index[mask_not_noise]]
y_known = df_routing.loc[mask_not_noise,'Группа поддержки'].values

knn = KNeighborsClassifier(n_neighbors=8,metric='cosine',n_jobs=-1)
knn.fit(X_known,y_known)

X_noise = vectors[df_routing.index[mask_noise]]
noise_predictions = knn.predict(X_noise)

df_routing.loc[mask_noise,'predicted_group'] = noise_predictions